# Sentence Trees
A treebank is English parsed into trees: every sentence is nested brackets, `(S (NP (DT The) (NN cat)) (VP (VBZ sits)))`. This notebook reads a real treebank, `VBGinTASA.dat`, with **one SNOBOL4 pattern**. While the pattern matches, small actions inside it keep a stack of lists, so the tree is built as the text is read, and the pattern counts every kind of phrase it sees. Run the cells in order.

## 1. Lists
The stack is made of lists, so first a small list library written in SNOBOL4: a `list` record with `ListAppend`, `ListInsert`, `ListRemove` and the rest.

In [ ]:
*------------------------------------------------------------------------------
                DATA('list(n,a)')
                DEFINE('ListValue(x,place)')
                DEFINE('ListName(x,place)')
                DEFINE('ListAppend(x,item)')
                DEFINE('ListPrepend(x,item)')
                DEFINE('ListInsert(x,item,place)a,i,size')
                DEFINE('ListRemove(x,place)i,size')
                DEFINE('ListPop(x)')
                DEFINE('ListSize(x)')                                   :(ListEnd)
*------------------------------------------------------------------------------
ListValue       place           =   LT(place, 0) n(x) + place
                ListValue       =   a(x)[place]                         :S(RETURN)F(FRETURN)
ListName        place           =   LT(place, 0) n(x) + place
                ListName        =   .a(x)[place]                        :S(RETURN)F(FRETURN)
*------------------------------------------------------------------------------
ListAppend      ListAppend      =   ListInsert(x, item, n(x))           :(RETURN)
*------------------------------------------------------------------------------
ListPrepend     ListPrepend     =   ListInsert(x, item, 0)              :(RETURN)
*------------------------------------------------------------------------------
ListInsert      ListInsert      =   x
                size            =   ListSize(x)
                place           =   LT(place, 0) n(x) + place
                GE(place, 0)                                            :F(error)
                LE(place, n(x))                                         :F(error)
                LT(n(x), size)                                          :F(ListInsert4)
                i               =   n(x) + 1
ListInsert1     i               =   GT(i, place + 1) i - 1              :F(ListInsert2)
                a(x)[i]         =   a(x)[i - 1]                         :(ListInsert1)
ListInsert2     a(x)[i - 1]     =   item                                :(ListInsert9)
ListInsert4     a               =   ARRAY('0:' (IDENT(a(x)) 0, size * 2 - 1))
                i               =   -1
ListInsert5     i               =   LT(i, place - 1) i + 1              :F(ListInsert6)
                a[i]            =   a(x)[i]                             :(ListInsert5)
ListInsert6     a[i + 1]        =   item
ListInsert7     i               =   LT(i, n(x) - 1) i + 1               :F(ListInsert8)
                a[i + 1]        =   a(x)[i]                             :(ListInsert7)
ListInsert8     a(x)            =   a
ListInsert9     n(x)            =   n(x) + 1                            :(RETURN)
*------------------------------------------------------------------------------
ListRemove      place           =   LT(place, 0) n(x) + place
                GE(place, 0)                                            :F(FRETURN)
                LT(place, n(x))                                         :F(FRETURN)
                ListRemove      =   a(x)[place]
                i               =   place
ListRemove1     i               =   LT(i, n(x) - 1) i + 1               :F(ListRemove2)
                a(x)[i - 1]     =   a(x)[i]                             :(ListRemove1)
ListRemove2     a(x)[i]         =
                n(x)            =   n(x) - 1                            :(RETURN)
*------------------------------------------------------------------------------
ListPop         ListPop         =   ListRemove(x, -1)                   :S(RETURN)F(FRETURN)
*------------------------------------------------------------------------------
ListSize        ListSize        =   IDENT(a(x)) 0                       :S(RETURN)
                PROTOTYPE(a(x)) '0:' REM . ListSize
                ListSize        =   ListSize + 1                        :(RETURN)
ListEnd

## 2. The pattern, and the run
A `group` is a tag, then words and more groups; `*group` calls the pattern recursively, and `Push_list` and `Pop_list` run as the match moves forward. The last lines open `VBGinTASA.dat`, beside this notebook, and match the whole file.

In [ ]:
*------------------------------------------------------------------------------
                DEFINE('Init_list(vs)');    DEFINE('init_list(v)')
                DEFINE('Push_list(vs)');    DEFINE('push_list(v)')
                DEFINE('Push_item(vs)');    DEFINE('push_item(v)')
                DEFINE('Pop_list()');       DEFINE('pop_list()')
                DEFINE('Pop_final(vs)');    DEFINE('pop_final(v)')      :(StackEnd)
*------------------------------------------------------------------------------
Init_list       Init_list       =   EVAL("epsilon . *init_list(" vs ")")  :(RETURN)
Push_list       Push_list       =   EVAL("epsilon . *push_list(" vs ")")  :(RETURN)
Push_item       Push_item       =   EVAL("epsilon . *push_item(" vs ")")  :(RETURN)
Pop_list        Pop_list        =         epsilon . *pop_list()           :(RETURN)
Pop_final       Pop_final       =   EVAL("epsilon . *pop_final(" vs ")")  :(RETURN)
*------------------------------------------------------------------------------
init_list       $v              =
                tags            =   TABLE()
                stack           =   list()
                init_list       =   .dummy                          :(NRETURN)
*------------------------------------------------------------------------------
push_list       tags[v]         =   tags[v] + 1
                ListAppend(stack, list())
                ListAppend(ListValue(stack, -1), v)
                push_list       =   .dummy                          :(NRETURN)
*------------------------------------------------------------------------------
push_item       ListAppend(ListValue(stack, -1), v)
                push_item       =   .dummy                          :(NRETURN)
*------------------------------------------------------------------------------
pop_list        ListAppend(ListValue(stack, -2), ListPop(stack))
                pop_list        =   .dummy                          :(NRETURN)
*------------------------------------------------------------------------------
pop_final       $v              =   ListPop(stack)
                pop_final       =   .dummy                          :(NRETURN)
*------------------------------------------------------------------------------
StackEnd        delim           =   SPAN(' ' CHAR(10))
                word            =   NOTANY('( )' CHAR(10)) BREAK('( )' CHAR(10))
*------------------------------------------------------------------------------
                group           =   '('
+                                   word . tag Push_list('tag')
+                                   ARBNO(
+                                     delim
+                                     FENCE(
+                                       *group
+                                     | word . wrd Push_item('wrd')
+                                     )
+                                   )
+                                   Pop_list()
+                                   ')'
*------------------------------------------------------------------------------
                treebank        =   POS(0)
+                                   Init_list("'bank'")
+                                   Push_list("'BANK'")
+                                   ARBNO(
+                                     Push_list("'ROOT'")
+                                     ARBNO(*group FENCE)
+                                     delim
+                                     FENCE
+                                     Pop_list()
+                                   )
+                                   Pop_final("'bank'")
+                                   RPOS(0)
*------------------------------------------------------------------------------
        INPUT(.INPUT, 9, 'VBGinTASA.dat [-r200000]')
        src = INPUT                                  :F(nomatch)
        src ? treebank                               :F(nomatch)
        OUTPUT = 'matched bytes=' SIZE(src)          :(shown)
nomatch OUTPUT = 'Pattern match failed'
shown

## 3. What it counted
The pattern kept a count of every tag it read.

In [ ]:
        OUTPUT = tags['ROOT'] ' sentences, ' tags['NP'] ' noun phrases, ' tags['VP'] ' verb phrases'